<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Dailychallenge_W8D3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installation
%pip install -qU "mcp[cli]"
!python --version
!mcp --help | head -n 5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.0 MB/s eta 0:00:00
Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


In [ ]:
%%writefile server.py
import logging
import json
from mcp.server.fastmcp import FastMCP

# Logs vers stderr pour ne pas polluer le canal STDIO
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("weather-server")

mcp = FastMCP("WeatherDemo")

# Base de données météo statique
WEATHER_DATA = {
    "paris": {"temp_c": 21, "condition": "sunny"},
    "london": {"temp_c": 15, "condition": "cloudy"},
    "nyc": {"temp_c": 28, "condition": "clear"},
}

@mcp.tool()
def get_weather(city: str) -> dict:
    """Get the current weather for a given city."""
    logger.info(f"Tool called: get_weather with city={city}")
    city_lower = city.lower()
    if city_lower in WEATHER_DATA:
        return {"city": city, **WEATHER_DATA[city_lower]}
    else:
        return {"error": f"City '{city}' not found. Supported: {list(WEATHER_DATA.keys())}"}

@mcp.resource("cities://list")
def get_cities() -> str:
    """Return a newline-separated list of supported cities."""
    logger.info("Resource called: cities://list")
    return "\n".join(WEATHER_DATA.keys())

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing server.py


In [ ]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None
)

def extract_content(payload):
    """
    Extract human-readable text from MCP response payloads.

    The MCP SDK returns different object structures depending on the call:
    - `read_resource` returns a `ReadResourceResult` with a `contents` list.
      Each item in this list is a `TextResourceContents` object containing a `text` attribute.
    - `call_tool` returns a `CallToolResult` with a `content` list.
      Each item in this list contains a `text` attribute.

    This function safely traverses these common structures to retrieve the
    textual content, allowing the client to print results cleanly.
    """
    # Handle ReadResourceResult (has 'contents')
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    # Handle CallToolResult (has 'content')
    if hasattr(payload, "content"):
        content_list = payload.content
        if content_list and hasattr(content_list[0], "text"):
            return content_list[0].text
    # Fallback
    return str(payload)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List resources
            resources = await session.list_resources()
            print("=== Available Resources ===")
            for res in resources.resources:
                print(f"  - URI: {res.uri} (Name: {res.name})")

            # List tools
            tools = await session.list_tools()
            print("\n=== Available Tools ===")
            for tool in tools.tools:
                print(f"  - Name: {tool.name} (Description: {tool.description})")

            # Read the cities list resource
            print("\n=== Reading resource: cities://list ===")
            result_resource = await session.read_resource("cities://list")
            cities_text = extract_content(result_resource)
            print(f"  Supported cities:\n{cities_text}")

            # Call the weather tool
            print("\n=== Calling tool: get_weather(city='Paris') ===")
            result_tool = await session.call_tool("get_weather", arguments={"city": "Paris"})
            weather_data = extract_content(result_tool)
            print(f"  Weather result: {weather_data}")

if __name__ == "__main__":
    asyncio.run(run())

Writing client.py


In [ ]:
!python client.py

INFO:mcp.server.lowlevel.server:Processing request of type ListResourcesRequest
=== Available Resources ===
  - URI: cities://list (Name: get_cities)
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest

=== Available Tools ===
  - Name: get_weather (Description: Get the current weather for a given city.)

=== Reading resource: cities://list ===
INFO:mcp.server.lowlevel.server:Processing request of type ReadResourceRequest
INFO:weather-server:Resource called: cities://list
  Supported cities:
paris
london
nyc

=== Calling tool: get_weather(city='Paris') ===
INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest
INFO:weather-server:Tool called: get_weather with city=Paris
  Weather result: {
  "city": "Paris",
  "temp_c": 21,
  "condition": "sunny"
}
